In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 0.1601, -0.1700,  0.1219,  0.1037, -0.1343, -0.0087,  0.2134, -0.2570,
         -0.0922, -0.1122],
        [ 0.1677, -0.0619,  0.0654,  0.1294, -0.2395, -0.1273,  0.2289, -0.1289,
         -0.0284, -0.0567]], grad_fn=<AddmmBackward0>)

In [2]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [3]:
net=MLP()
net(X)

tensor([[-2.2923e-01,  1.3692e-01, -5.1877e-02,  2.6637e-01, -1.3786e-01,
         -1.0738e-01,  3.9743e-02, -2.3930e-01, -5.1896e-03, -1.6286e-04],
        [-9.3381e-02,  9.6489e-02, -9.0947e-02,  2.2717e-01,  1.4163e-02,
          4.0495e-02,  1.5885e-01, -3.8767e-02, -5.0468e-02,  2.9108e-02]],
       grad_fn=<AddmmBackward0>)

In [4]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X
net=MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.1626,  0.1657, -0.0629, -0.0299, -0.0292, -0.0967,  0.1959,  0.0037,
         -0.0431,  0.0691],
        [ 0.0630,  0.1758, -0.1468,  0.0028,  0.0253, -0.0220,  0.0821,  0.0128,
         -0.0451, -0.0020]], grad_fn=<AddmmBackward0>)

In [20]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以 及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [21]:
net=FixedHiddenMLP()
net(X)

tensor(-0.0607, grad_fn=<SumBackward0>)

In [24]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.6507, grad_fn=<SumBackward0>)